In [1]:
!pip install mamba-ssm --no-build-isolation
!pip install causal-conv1d>=1.4.0 --no-build-isolation
!pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 5.5 MB/s eta 0:00:0000:01
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 45.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 25.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 67.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 44.3 MB/s eta 0:00:00
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.2.post1-cp312-cp312-linux_x86_64.whl size=322288410 sha256=7a67070c1e7e99c95abd1319623f044e8a1b3fb46f774bfdea949f0a4fc79638
  Stored in directory: /root/.cache/pip/wheels/da/67/03/99148d6eeaa4ec2855d71295ac83bcbc8ba

In [2]:
import os
import torch
import torch.nn as nn
import numpy as np
from scipy.signal import resample
from scipy.io import loadmat
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import autocast
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import roc_auc_score, f1_score, classification_report, average_precision_score, hamming_loss, accuracy_score
from mamba_ssm import Mamba
from tqdm.notebook import tqdm

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm_x = torch.mean(x ** 2, dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(norm_x + self.eps)
        return self.weight * x_normed

class BiMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.forward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.backward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        
    def forward(self, x):
        normed_x = self.norm(x)
        fwd_out = self.forward_mamba(normed_x)
        bwd_out = self.backward_mamba(normed_x.flip(dims=[1])).flip(dims=[1])
        return x + fwd_out + bwd_out

class AdvancedECGMamba(nn.Module):
    def __init__(self, in_channels=12, d_model=256, d_state=32, num_classes=5, num_layers=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, d_model // 2, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(d_model // 2),
            nn.GELU(),
            nn.Conv1d(d_model // 2, d_model, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(d_model),
            nn.GELU()
        )
        self.layers = nn.ModuleList([BiMambaBlock(d_model=d_model, d_state=d_state) for _ in range(num_layers)])
        self.final_norm = RMSNorm(d_model)
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x):
        x = self.stem(x) 
        x = x.transpose(1, 2)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        x_max, _ = torch.max(x, dim=1)
        return self.head(x_max)

def evaluate_mamba_advanced(model, dataloader, device):
    model.eval()
    all_probs, all_targets = [], []
    
    with torch.no_grad():
        for signals, labels in dataloader:
            signals = signals.to(device, non_blocking=True)
            with autocast('cuda', dtype=torch.bfloat16):
                probs = torch.sigmoid(model(signals)) 
            all_probs.append(probs.cpu().float().numpy())
            all_targets.append(labels.numpy())
            
    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    
    aucs, auprcs = [], []
    superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
    
    for i in range(len(superclasses)):
        if len(np.unique(all_targets[:, i])) == 2:
            aucs.append(roc_auc_score(all_targets[:, i], all_probs[:, i]))
            auprcs.append(average_precision_score(all_targets[:, i], all_probs[:, i]))
            
    macro_auc = np.mean(aucs) if len(aucs) > 0 else 0.0
    macro_auprc = np.mean(auprcs) if len(auprcs) > 0 else 0.0
        
    return macro_auc, macro_auprc

In [3]:
# ==========================================
# CPSC DATASET (DYNAMIC CROP & PAD LOADER)
# ==========================================
class CPSCDataset(Dataset):
    def __init__(self, data_path, target_hz=100, original_hz=500):
        super().__init__()
        self.data_path = data_path
        self.superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
        self.target_length = target_hz * 10 # Exactly 1000 samples (10s)
        self.target_hz = target_hz
        self.original_hz = original_hz
        
        # SNOMED-CT Mapping to PTB-XL Superclasses
        self.snomed_mapping = {
            '426783006': 'NORM', '426177001': 'NORM', '427084000': 'NORM', '427393009': 'NORM',
            '164865005': 'MI', '164861001': 'MI', '54329005': 'MI',
            '164934002': 'STTC', '59931005': 'STTC', '164917005': 'STTC', '111975006': 'STTC',
            '164909002': 'CD', '733534002': 'CD', '59118001': 'CD', '713427006': 'CD', 
            '713426002': 'CD', '445118002': 'CD', '270492004': 'CD', '164947007': 'CD', '6374002': 'CD',
            '164873001': 'HYP', '370355005': 'HYP'
        }

        self.files = [f[:-4] for f in os.listdir(data_path) if f.endswith('.mat') and not f.startswith('.')]
        
        self.signals = []
        self.labels = []
        self.skipped_files = 0
        
        print(f"Pre-loading CPSC 2018 | Resampling 500Hz -> {target_hz}Hz | Enforcing 10s Sequence...")
        for f in tqdm(self.files, desc="Processing ECGs"):
            mat_path = os.path.join(self.data_path, f + '.mat')
            hea_path = os.path.join(self.data_path, f + '.hea')
            
            try:
                # 1. Read Native MATLAB Signal
                mat_data = loadmat(mat_path)
                key = [k for k in mat_data.keys() if not k.startswith('__')][0]
                signal = np.array(mat_data[key], dtype=np.float32)
                
                if signal.shape[0] == 12:
                    signal = signal.T 
                
                signal = signal / 1000.0 # Standardize voltage
                signal = np.nan_to_num(signal)
                
                # --- THE FIX: DYNAMIC TEMPORAL RESAMPLING ---
                # Calculate exact samples to keep the time-scale accurate
                time_ratio = self.target_hz / self.original_hz
                dynamic_samples = int(signal.shape[0] * time_ratio)
                signal = resample(signal, dynamic_samples, axis=0) 
                
                # --- ENFORCE 10-SECOND SEQUENCE ---
                if signal.shape[0] > self.target_length:
                    # Signal is too long -> Center Crop
                    start = (signal.shape[0] - self.target_length) // 2
                    signal = signal[start : start + self.target_length, :]
                elif signal.shape[0] < self.target_length:
                    # Signal is too short -> Symmetrical Zero Padding
                    pad_total = self.target_length - signal.shape[0]
                    pad_top = pad_total // 2
                    pad_bottom = pad_total - pad_top
                    signal = np.pad(signal, ((pad_top, pad_bottom), (0, 0)), mode='constant')
                
                # Convert back to (Channels, Time)
                signal_tensor = torch.tensor(signal, dtype=torch.float32).transpose(0, 1)
                
                # 2. Extract Labels manually bypassing wfdb errors
                with open(hea_path, 'r', encoding='utf-8-sig', errors='ignore') as text_file:
                    lines = text_file.readlines()
                
                dx_str = ""
                for line in lines:
                    if line.startswith('#Dx:') or line.startswith('# Dx:'):
                        dx_str = line.split(':')[1].strip()
                        break
                
                if not dx_str:
                    self.skipped_files += 1
                    continue
                
                label = np.zeros(len(self.superclasses))
                for code in dx_str.split(','):
                    code = code.strip()
                    if code in self.snomed_mapping:
                        idx = self.superclasses.index(self.snomed_mapping[code])
                        label[idx] = 1
                        
                self.signals.append(signal_tensor)
                self.labels.append(torch.tensor(label, dtype=torch.float32))
                
            except Exception as e:
                self.skipped_files += 1
                continue

        print(f"Successfully loaded {len(self.signals)} records. Skipped {self.skipped_files} corrupted files.")

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

In [4]:
# 1. Setup Directories
CPSC_DIR = "/kaggle/input/datasets/organizations/physionet/china-physiological-signal-challenge-in-2018/Training_WFDB"

# 2. Load and Split the Dataset (80% Train, 20% Validation)
full_dataset = CPSCDataset(data_path=CPSC_DIR, target_hz=100)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

# num_workers=0 remains critical to safeguard Kaggle RAM
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Training on {train_size} records. Validating on {val_size} records.")

# 3. Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AdvancedECGMamba(in_channels=12, d_model=256, d_state=32, num_classes=5, num_layers=6).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
criterion = nn.BCEWithLogitsLoss()

# 4. The Training Loop with Live Validation
epochs = 30
max_norm = 1.0

print("Starting training from scratch on CPSC 2018...")
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    valid_batches = 0
    
    for i, (signals, labels) in enumerate(train_dataloader):
        signals, labels = signals.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True) 
        
        with autocast('cuda', dtype=torch.bfloat16):
            outputs = model(signals)
            loss = criterion(outputs, labels)
            
        if torch.isnan(loss):
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        
        optimizer.step()
        total_loss += loss.item()
        valid_batches += 1
        
    scheduler.step()
    avg_train_loss = total_loss / max(valid_batches, 1)
    
    # Live Validation check at the end of each epoch
    val_auc, val_auprc = evaluate_mamba_advanced(model, val_dataloader, device)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val AUC: {val_auc:.4f} | Val AUPRC: {val_auprc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# Optional: Save the weights when done
torch.save(model.state_dict(), '/kaggle/working/mamba_cpsc_weights.pth')
print("Model saved to /kaggle/working/mamba_cpsc_weights.pth")

Pre-loading CPSC 2018 | Resampling 500Hz -> 100Hz | Enforcing 10s Sequence...


Processing ECGs:   0%|          | 0/6877 [00:00<?, ?it/s]

Successfully loaded 6877 records. Skipped 0 corrupted files.
Training on 5501 records. Validating on 1376 records.
Starting training from scratch on CPSC 2018...
Epoch [1/30] | Train Loss: 0.1566 | Val AUC: 0.9394 | Val AUPRC: 0.7707 | LR: 0.000976
Epoch [2/30] | Train Loss: 0.1062 | Val AUC: 0.9617 | Val AUPRC: 0.8173 | LR: 0.000905
Epoch [3/30] | Train Loss: 0.0911 | Val AUC: 0.9623 | Val AUPRC: 0.8415 | LR: 0.000794
Epoch [4/30] | Train Loss: 0.0832 | Val AUC: 0.9613 | Val AUPRC: 0.8427 | LR: 0.000655
Epoch [5/30] | Train Loss: 0.0733 | Val AUC: 0.9661 | Val AUPRC: 0.8549 | LR: 0.000501
Epoch [6/30] | Train Loss: 0.0629 | Val AUC: 0.9693 | Val AUPRC: 0.8589 | LR: 0.000346
Epoch [7/30] | Train Loss: 0.0520 | Val AUC: 0.9712 | Val AUPRC: 0.8724 | LR: 0.000207
Epoch [8/30] | Train Loss: 0.0433 | Val AUC: 0.9720 | Val AUPRC: 0.8771 | LR: 0.000096
Epoch [9/30] | Train Loss: 0.0330 | Val AUC: 0.9717 | Val AUPRC: 0.8833 | LR: 0.000025
Epoch [10/30] | Train Loss: 0.0268 | Val AUC: 0.9736 | 